# WaterOnt: LLM comparison run

Runs the two LLM cells of Table 5 (flat-text RAG and WaterOnt GraphRAG), plus every other result in Sections 5 and 6.

**Before you start**
1. Runtime → **Disconnect and delete runtime** (a fresh machine avoids leftover packages).
2. Add your key in **Secrets** (key icon, left sidebar): name `ANTHROPIC_API_KEY`, value your API key (starts with `sk-ant-api`), notebook access on.
3. Have `WaterOnt-full-repo.zip` on your computer; step 2 asks for it.
4. Run the cells in order. Step 1 may restart the runtime once; if it does, continue from step 2. **Do not continue past step 3 unless it prints `API replied: OK`.**

In [ ]:
# 1. Install packages (keeps NumPy compatible with Colab's SciPy)
import os, subprocess, sys
os.chdir("/content")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "numpy>=2.0,<2.3", "wntr", "rdflib", "anthropic"], check=True)

on_disk = subprocess.run([sys.executable, "-c", "import numpy; print(numpy.__version__)"],
                         capture_output=True, text=True).stdout.strip()
import numpy
if numpy.__version__ != on_disk:
    print(f"NumPy changed ({numpy.__version__} -> {on_disk}): restarting. Continue from step 2.")
    os.kill(os.getpid(), 9)
print("Step 1 done. NumPy", numpy.__version__)

In [ ]:
# 2. Get the code from WaterOnt-full-repo.zip (no GitHub needed)
import os, sys, shutil, zipfile
from google.colab import files
os.chdir("/content")
uploaded = files.upload()                       # choose WaterOnt-full-repo.zip
shutil.rmtree("/content/WaterOnt", ignore_errors=True)
with zipfile.ZipFile(next(iter(uploaded))) as z:
    z.extractall("/content")                    # creates /content/WaterOnt
os.chdir("/content/WaterOnt")
if "/content/WaterOnt" not in sys.path:
    sys.path.insert(0, "/content/WaterOnt")
for name in [m for m in sys.modules if m.startswith("wateront")]:
    del sys.modules[name]                       # forget any older version loaded earlier

from wateront.baselines import TableRulesBaseline
from wateront.config import Config
print("Step 2 done. Code ready; LLM model:", Config().llm_model)

In [ ]:
# 3. API key and a one-line test call. Must print "API replied: OK" before you continue.
import os, anthropic
from google.colab import userdata
key = userdata.get("ANTHROPIC_API_KEY").strip().strip("\"'")
print("Key check | length:", len(key), "| starts with sk-ant-api:", key.startswith("sk-ant-api"))
os.environ["ANTHROPIC_API_KEY"] = key

reply = anthropic.Anthropic().messages.create(
    model=Config().llm_model, max_tokens=10,
    messages=[{"role": "user", "content": "Reply with OK"}])
print("Step 3 done. API replied:", reply.content[0].text)

In [ ]:
# 4. Smoke test: one question for each LLM system (a few seconds)
from wateront.pipeline import build_knowledge_graph
from wateront.graphrag import GraphRAG
from wateront.baselines import FlatRAGBaseline

cfg = Config(seed=42)
res = build_knowledge_graph("data/networks/L-TOWN.inp", cfg, verbose=False)
question = "Which unmonitored hydraulically critical mains have intolerable risk?"

kg_answer = GraphRAG(res.kg, cfg, use_llm=True, strict=True).ask(question)   # strict: errors are not hidden
print("WITH KG + LLM:", len(kg_answer.recommendations), "pipes; generated by:", kg_answer.generator, "; unsupported citations:", kg_answer.unsupported_citations)
print(kg_answer.text[:500], "\n")

flat_answer = FlatRAGBaseline(res.kg, cfg, use_llm=True).ask(question)
print("WITHOUT KG + LLM:", len(flat_answer.recommendations), "pipes; unsupported citations:", flat_answer.unsupported_citations)
print(flat_answer.text[:500])

Both answers should list pipes. If the second shows 0 pipes and its text is not a JSON list, stop and send the output of step 4.

In [ ]:
# 5. Full run: all results of Sections 5 and 6, including the two LLM cells (about 15 minutes)
import subprocess, sys
run = subprocess.run([sys.executable, "paper/experiments.py"], capture_output=True, text=True,
                     cwd="/content/WaterOnt")
print(run.stdout[-2000:])
if run.returncode != 0:
    print(run.stderr[-3000:])
    raise RuntimeError("experiments.py failed; send the error above")
print("Step 5 done.")

In [ ]:
# 6. Results, and the two rows for Table 5
import json, pandas as pd
meta = json.load(open("paper/results/meta.json"))
print("LLM cells run:", meta.get("llm_cells_run"), "| LLM planning failures (scored as empty answers):",
      meta.get("llm_planning_failures", 0))
table = pd.read_csv("paper/results/comparison_summary.csv").set_index("system").round(3)
display(table)

rows = {"No KG, LLM (flat-text RAG)": ("Flat text (RAG)", "no"), "KG, LLM": (r"\textsc{WaterOnt}", "yes")}
for system, (name, path) in rows.items():
    if system in table.index:
        r = table.loc[system]
        print(rf"\quad {name} & {r.decision_f1:.3f} & {r.policy_recall:.3f} & {r.traceability:.3f} & "
              rf"{r.evidence_consistency:.3f} & {r.unsupported_citations:.2f} & {path} \\")

In [ ]:
# 7. Download all results
import shutil
from google.colab import files
shutil.make_archive("/content/llm_results", "zip", "/content/WaterOnt/paper", "results")
files.download("/content/llm_results.zip")